# system_tai Phase 2.5 — ground-truth KIS benchmark

This notebook evaluates only human-verified labels. The checked-in template contains drafts, so it cannot produce quality metrics until a reviewer supplies real `frame_id` positives. `/kaggle/working` is ephemeral: download or externally version reports that must survive the session. No dataset artifact is copied.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

import torch

REPOSITORY_ROOT = Path('/kaggle/working/AI_Challenge_HCM')
SYSTEM_ROOT = REPOSITORY_ROOT / 'systems/system_tai'
sys.path.insert(0, str(SYSTEM_ROOT / 'src'))
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path(
    os.environ.get(
        'SYSTEM_TAI_BENCHMARK_OUTPUT',
        '/kaggle/working/system_tai_outputs/kis_benchmark',
    )
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BENCHMARK_PATH = Path(
    os.environ.get(
        'SYSTEM_TAI_BENCHMARK_FILE',
        str(SYSTEM_ROOT / 'config/kis_benchmark.example.yaml'),
    )
)
DEVICE_REQUEST = os.environ.get('SYSTEM_TAI_DEVICE', 'auto').lower()
ALLOW_MODEL_DOWNLOAD = os.environ.get('SYSTEM_TAI_ALLOW_MODEL_DOWNLOAD', '0') == '1'
if DEVICE_REQUEST not in {'auto', 'cpu', 'cuda'}:
    raise ValueError(f'unsupported device: {DEVICE_REQUEST}')
if DEVICE_REQUEST == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA was requested but is unavailable')
DEVICE = 'cuda' if DEVICE_REQUEST == 'auto' and torch.cuda.is_available() else DEVICE_REQUEST
if DEVICE == 'auto':
    DEVICE = 'cpu'
device_summary = {
    'device': DEVICE,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None,
}
print(device_summary)
print('GPU changes latency only; it does not change retrieval semantics.')

## Bounded dataset discovery

Resolve one dataset root from the shallow `/kaggle/input/datasets/*/*` candidates, then construct the three audited video paths directly.

In [ ]:
REQUIRED_FAMILIES = (
    Path('map-keyframes-aic25-b1/map-keyframes'),
    Path('clip-features-32-aic25-b1/clip-features-32'),
    Path('keyframes/keyframes'),
)
candidate_roots = sorted(
    path
    for path in (INPUT_ROOT / 'datasets').glob('*/*')
    if path.is_dir()
)
valid_roots = [
    root
    for root in candidate_roots
    if all((root / family).is_dir() for family in REQUIRED_FAMILIES)
]
if len(valid_roots) != 1:
    raise RuntimeError(
        'expected exactly one Dataset_AIC2026 root, '
        f'found {len(valid_roots)}: {valid_roots}'
    )
DATASET_ROOT = valid_roots[0]
print(f'dataset root resolved: {DATASET_ROOT}')

In [ ]:
VIDEO_IDS = ('L21_V001', 'L21_V002', 'L22_V001')
videos = []
keyframe_directories = {}
for video_id in VIDEO_IDS:
    group = video_id.split('_', 1)[0]
    mapping = DATASET_ROOT / 'map-keyframes-aic25-b1/map-keyframes' / f'{video_id}.csv'
    features = DATASET_ROOT / 'clip-features-32-aic25-b1/clip-features-32' / f'{video_id}.npy'
    keyframes = DATASET_ROOT / 'keyframes/keyframes' / f'Keyframes_{group}' / 'keyframes' / video_id
    missing = [str(path) for path in (mapping, features, keyframes) if not path.exists()]
    if missing:
        raise FileNotFoundError(f'missing required paths for {video_id}: {missing}')
    videos.append(
        {
            'video_id': video_id,
            'mapping_csv_path': str(mapping),
            'clip_npy_path': str(features),
        }
    )
    keyframe_directories[video_id] = keyframes
MANIFEST_PATH = OUTPUT_ROOT / 'feature_manifest.json'
MANIFEST_PATH.write_text(json.dumps({'videos': videos}, indent=2) + '\n', encoding='utf-8')
print(f'manifest created: {MANIFEST_PATH}')

## Registry and benchmark validation

Drafts are validated but excluded from scoring. A verified query without a human-authored positive is invalid.

In [ ]:
from system_tai.evaluation.benchmark_validator import BenchmarkValidator
from system_tai.features.btc_clip_store import FeatureStoreRegistry

registry = FeatureStoreRegistry.from_manifest(MANIFEST_PATH)
print(f'registry loaded with row count: {registry.total_rows}')
validation = BenchmarkValidator().validate_file(BENCHMARK_PATH, registry, include_drafts=True)
validation_summary = {
    'valid': validation.valid,
    'errors': [asdict(issue) for issue in validation.errors],
    'warnings': [asdict(issue) for issue in validation.warnings],
    'verified': len(validation.verified_queries),
    'drafts': len(validation.draft_queries),
}
print(validation_summary)
if not validation.valid:
    raise ValueError('benchmark validation failed; fix structured errors before evaluation')

In [ ]:
from system_tai.evaluation.kis_benchmark import KISBenchmarkEvaluator
from system_tai.evaluation.reports import write_benchmark_reports
from system_tai.features.query_encoder import OpenAIClipTextEncoder
from system_tai.retrieval.vector_search import ExactNumpyRetriever

encoder = OpenAIClipTextEncoder(device=DEVICE, allow_model_download=ALLOW_MODEL_DOWNLOAD)
retriever = ExactNumpyRetriever(registry, encoder, chunk_size=4096)
print(f'text encoder loaded: model={encoder.identifiers.get("model")} device={DEVICE}')
report = None
if validation.verified_queries:
    report = KISBenchmarkEvaluator().evaluate(
        validation, retriever, top_ks=(1, 5, 20, 50, 100)
    )
    report_paths = write_benchmark_reports(report, OUTPUT_ROOT)
    compact_metrics = [asdict(metric) for metric in report.query_metrics]
    display(compact_metrics)
    print(f'evaluation completed; reports written: {report_paths}')
else:
    print('evaluation state=no_verified_queries; no quality metrics were fabricated')

## Manual annotation helper

The helper lists bounded candidates and external image paths. It never embeds an image or marks a frame relevant. A human must review and explicitly promote labels to `verified`.

In [ ]:
from system_tai.common.schemas import KISQuery
from system_tai.evaluation.annotation import (
    build_annotation_candidates,
    write_draft_annotation_review,
)

if validation.draft_queries:
    draft = validation.draft_queries[0]
    result = retriever.retrieve(
        KISQuery(query_id=draft.query_id, text=draft.text, top_k=20)
    )
    candidates = build_annotation_candidates(
        result, keyframe_directories, limit=20
    )
    display([asdict(candidate) for candidate in candidates])
    review_path = write_draft_annotation_review(
        draft,
        candidates,
        OUTPUT_ROOT / f'annotation_{draft.query_id}.json',
    )
    print(f'draft annotation review written: {review_path}')
else:
    print('no draft query available for annotation helper')

In [ ]:
print({
    'benchmark_valid': validation.valid,
    'evaluation_state': (
        'no_verified_queries' if report is None else report.evaluation_state
    ),
    'invalid_query_count': validation.invalid_query_count,
    'verified_queries_evaluated': 0 if report is None else report.evaluated_query_count,
    'draft_queries_excluded': len(validation.draft_queries),
    'canonical_unsuppressed': True,
    'output_directory': str(OUTPUT_ROOT),
    'next_action': (
        'Human-verify official frame_id labels, rerun, then download reports '
        'from ephemeral Kaggle working storage.'
    ),
})